# Neo4j Aura BC Private Link Connectivity Test

**Status: Working** — Validated 2026-03-20. Databricks serverless connects to Neo4j Aura BC over Private Link via Application Gateway v2 L4 TCP proxy.

```
Databricks Serverless --> NCC Private Endpoint --> Private Link tunnel
    --> Application Gateway v2 (L4 TCP passthrough, port 7687) --> Neo4j Aura BC
```

**Prerequisites:**
- Azure infrastructure deployed (`setup_azure.py phase1` then `phase2`)
- App Gateway public IP added to Aura BC allowlist (`manage_ip_allowlist.py add`)
- NCC created and private endpoint rule added (`deploy.py create-ncc`, `create-pe-rule`)
- Connection approved (`deploy.py approve`) and NCC attached to this workspace (`attach-ncc`)
- Secrets stored in a Databricks secret scope (`deploy.py setup-secrets`)

**Key constraints:**
- Must run on **serverless** compute — NCC PE rules are not active on classic clusters
- Driver must set `max_connection_lifetime < 300` and `liveness_check_timeout < 300` due to Azure Private Link ~5 min idle timeout

**Protocol support:**
- Tests 1-2 validate `bolt+s://` (single connection, no routing table)
- Test 3 validates `neo4j+s://` (client-side routing with routing table discovery) — requires `deploy.py update-pe-domains` to add routing table hostnames to the NCC PE rule

In [ ]:
%pip install neo4j>=5.20.0

---

## Setup: Create Secret Scope

Run this once from your terminal to store Neo4j credentials and domain:

```bash
uv run python deploy.py setup-secrets --profile <your-profile>
```

Or manually via the Databricks CLI:

```bash
databricks secrets create-scope neo4j-appgw-poc
databricks secrets put-secret neo4j-appgw-poc password --string-value '<NEO4J_PASSWORD>'
databricks secrets put-secret neo4j-appgw-poc domain --string-value '<AURA_FQDN>'
```

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Credentials and domain from Databricks secrets
SCOPE_NAME = "neo4j-appgw-poc"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = dbutils.secrets.get(SCOPE_NAME, "password")

# The NCC private endpoint rule domain must be the real Aura FQDN so that the
# TLS SNI hostname matches what Aura BC expects. Using a made-up private domain
# causes Aura to reject the TLS handshake.
NEO4J_DOMAIN = dbutils.secrets.get(SCOPE_NAME, "domain")

# bolt+s:// is required for Aura BC through the Application Gateway.
# neo4j+s:// performs routing table discovery which bypasses the proxy entirely.
NEO4J_URI = f"bolt+s://{NEO4J_DOMAIN}:7687"

# Keepalive settings for Azure Private Link ~5 min idle timeout
MAX_CONNECTION_LIFETIME = 240  # seconds, must be < 300
LIVENESS_CHECK_TIMEOUT = 120  # seconds, must be < 300
CONNECTION_ACQUISITION_TIMEOUT = 30  # seconds

print("Configuration loaded:")
print(f"  Domain: {NEO4J_DOMAIN}")
print(f"  URI:    {NEO4J_URI}")
print(f"  User:   {NEO4J_USER}")
print(f"  Max connection lifetime: {MAX_CONNECTION_LIFETIME}s")
print(f"  Liveness check timeout:  {LIVENESS_CHECK_TIMEOUT}s")

---

## Test 1: TCP Connectivity

Verifies the network path is open from serverless compute to the Application Gateway
over Private Link on port 7687 (Bolt).

In [ ]:
import subprocess
import time

print("=" * 60)
print("TEST: TCP Connectivity over Private Link")
print("=" * 60)
print(f"\nTarget: {NEO4J_DOMAIN}:7687")

try:
    start = time.time()
    result = subprocess.run(
        ["nc", "-zv", NEO4J_DOMAIN, "7687"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10,
    )
    elapsed = (time.time() - start) * 1000
    output = (result.stdout.decode() + result.stderr.decode()).strip()

    if result.returncode == 0:
        print(f"\n[PASS] TCP connection established in {elapsed:.1f}ms")
        if output:
            print(f"  Raw: {output}")
    else:
        print(f"\n[FAIL] Cannot reach {NEO4J_DOMAIN}:7687")
        print(f"  Output: {output}")
        print("\n  Check that:")
        print("  - NCC private endpoint status is ESTABLISHED")
        print("  - NCC is attached to this workspace")
        print("  - Domain in NCC rule matches NEO4J_DOMAIN above")

except Exception as e:
    print(f"\n[FAIL] Error: {e}")

---

## Test 2: Neo4j Driver Connectivity

Connects using the Neo4j Python driver over `bolt+s://` with keepalive settings
tuned for the Azure Private Link idle timeout (~300s). Authenticates and runs a test query.

In [ ]:
from neo4j import GraphDatabase
import time

print("=" * 60)
print("TEST: Neo4j Driver over Private Link")
print("=" * 60)
print(f"\nURI: {NEO4J_URI}")

try:
    start = time.time()
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
        max_connection_lifetime=MAX_CONNECTION_LIFETIME,
        liveness_check_timeout=LIVENESS_CHECK_TIMEOUT,
        connection_acquisition_timeout=CONNECTION_ACQUISITION_TIMEOUT,
    )
    driver.verify_connectivity()
    connect_ms = (time.time() - start) * 1000
    print(f"\n[PASS] Connected and authenticated in {connect_ms:.1f}ms")

    # Test query
    records, _, _ = driver.execute_query(
        "RETURN 'Connected over Private Link via bolt+s' AS message"
    )
    print(f"[PASS] Query result: {records[0]['message']}")

    # Server info
    records, _, _ = driver.execute_query(
        "CALL dbms.components() YIELD name, versions RETURN name, versions"
    )
    for record in records:
        print(f"\n  Server: {record['name']} {record['versions']}")

    driver.close()
    total_ms = (time.time() - start) * 1000

    print(f"\n  Connection: {connect_ms:.1f}ms")
    print(f"  Total:      {total_ms:.1f}ms")
    print("\n" + "=" * 60)
    print("PRIVATE LINK CONNECTIVITY VERIFIED")
    print("=" * 60)

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("\n  Check that:")
    print("  - App Gateway public IP is in Aura BC allowlist (manage_ip_allowlist.py list)")
    print("  - Password in secret scope is correct")
    print("  - URI uses bolt+s:// (not neo4j+s://)")
    print("  - App Gateway L4 listener is active on port 7687 (setup_azure.py status)")

---

## Test 3: neo4j+s:// Protocol with Client-Side Routing

Tests `neo4j+s://` protocol through the Private Link tunnel. Unlike `bolt+s://`, this protocol
fetches a routing table from the server and distributes queries across cluster members (readers,
writers, routers).

**Requires:** Run `deploy.py update-pe-domains` first to add routing table hostnames to the NCC
PE rule. Without this, the driver connects on the initial FQDN but fails when attempting
connections to routing table member hostnames (NCC won't intercept their DNS).

The App Gateway L4 TCP listener operates in passthrough mode — it forwards raw bytes without
terminating TLS. The SNI hostname in the driver's TLS ClientHello reaches Aura untouched,
allowing Aura to route to the correct cluster member.

In [ ]:
from neo4j import GraphDatabase
import time

print("=" * 60)
print("TEST 3: neo4j+s:// Protocol with Client-Side Routing")
print("=" * 60)

neo4j_uri = f"neo4j+s://{NEO4J_DOMAIN}:7687"
print(f"\nURI: {neo4j_uri}")

driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(NEO4J_USER, NEO4J_PASSWORD),
    max_connection_lifetime=MAX_CONNECTION_LIFETIME,
    liveness_check_timeout=LIVENESS_CHECK_TIMEOUT,
    connection_acquisition_timeout=CONNECTION_ACQUISITION_TIMEOUT,
)

try:
    start = time.time()
    driver.verify_connectivity()
    connect_ms = (time.time() - start) * 1000
    print(f"\n[PASS] Connected with neo4j+s:// in {connect_ms:.1f}ms")

    # ---- Routing table inspection ----
    print("\n--- Routing Table ---")
    pool = driver._pool
    routing_members = {"routers": [], "readers": [], "writers": []}
    if hasattr(pool, "routing_tables"):
        for db, table in pool.routing_tables.items():
            print(f"  Database: {db}, TTL: {table.ttl}s")
            for role in ("routers", "readers", "writers"):
                addrs = getattr(table, role, [])
                routing_members[role] = [f"{a[0]}:{a[1]}" for a in addrs]
                print(f"  {role}: {routing_members[role]}")
    else:
        print("  WARNING: Could not access routing tables")

    # ---- Round 1: Initial read distribution ----
    print("\n--- Round 1: Read Distribution (10 queries) ---")
    read_servers_r1 = []
    for i in range(10):
        records, summary, keys = driver.execute_query(
            "RETURN $i AS n",
            i=i,
            routing_="r",
        )
        server = f"{summary.server.address[0]}:{summary.server.address[1]}"
        read_servers_r1.append(server)
        print(f"  Read {i+1:2d}: server={server}")

    # ---- Write distribution ----
    print("\n--- Write Distribution (5 queries) ---")
    write_servers = []
    for i in range(5):
        records, summary, keys = driver.execute_query(
            "RETURN $i AS n",
            i=i,
            routing_="w",
        )
        server = f"{summary.server.address[0]}:{summary.server.address[1]}"
        write_servers.append(server)
        print(f"  Write {i+1}: server={server}")

    # ---- Wait for routing table TTL refresh ----
    print("\n--- Waiting 12 seconds for routing table TTL refresh (TTL=10s) ---")
    time.sleep(12)

    # ---- Check if routing table changed ----
    print("\n--- Routing Table After Refresh ---")
    if hasattr(pool, "routing_tables"):
        for db, table in pool.routing_tables.items():
            print(f"  Database: {db}, TTL: {table.ttl}s")
            for role in ("routers", "readers", "writers"):
                addrs = getattr(table, role, [])
                print(f"  {role}: {[f'{a[0]}:{a[1]}' for a in addrs]}")

    # ---- Round 2: Read distribution after TTL refresh ----
    print("\n--- Round 2: Read Distribution After Refresh (10 queries) ---")
    read_servers_r2 = []
    for i in range(10):
        records, summary, keys = driver.execute_query(
            "RETURN $i AS n",
            i=i,
            routing_="r",
        )
        server = f"{summary.server.address[0]}:{summary.server.address[1]}"
        read_servers_r2.append(server)
        print(f"  Read {i+1:2d}: server={server}")

    # ---- Summary ----
    all_read_servers = set(read_servers_r1 + read_servers_r2)
    all_write_servers = set(write_servers)

    print("\n" + "=" * 60)
    print("SUMMARY")
    print("=" * 60)
    print(f"\n  Read servers (round 1):  {sorted(set(read_servers_r1))}")
    print(f"  Read servers (round 2):  {sorted(set(read_servers_r2))}")
    print(f"  Write servers:           {sorted(all_write_servers)}")
    print(f"  Total unique servers:    {len(all_read_servers | all_write_servers)}")
    print(f"  Routing table members:   {len(routing_members['routers'])}")

    reads_changed = set(read_servers_r1) != set(read_servers_r2)
    multi_read = len(all_read_servers) > 1
    writes_consistent = len(all_write_servers) == 1

    print(f"\n  Read distribution:       {'MULTI-SERVER' if multi_read else 'SINGLE SERVER'}")
    print(f"  Reads changed after TTL: {'YES' if reads_changed else 'NO'}")
    print(f"  Writes consistent:       {'YES (single writer)' if writes_consistent else 'MULTIPLE WRITERS'}")

    if multi_read:
        print(f"\n[PASS] CLIENT-SIDE ROUTING VERIFIED — reads distributed across servers")
    else:
        print(f"\n[PASS] neo4j+s:// connected and queries executed through the tunnel")
        print(f"  Reads went to a single server — Aura may be routing all traffic")
        print(f"  through one member behind the Private Link endpoint. This is")
        print(f"  expected when the tunnel terminates at a single IP.")

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("\n  Check that:")
    print("  - deploy.py update-pe-domains has been run")
    print("  - NCC changes have propagated (~5 min after update)")
    print("  - Serverless compute was restarted after the NCC update")
    print("  - App Gateway public IP is in Aura BC allowlist")

finally:
    driver.close()